In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))
import os
from dotenv import load_dotenv

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from openai import AsyncOpenAI, OpenAI

from openagv.core import SqliteAssetBin
from openagv.modules.audio import DeepgramAnalyzer

# Load environment variables from .env
load_dotenv()

async_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

chat_completion_service = OpenAIChatCompletion(
    service_id="my-service-id",
    ai_model_id="openai/gpt-4o-mini",
    async_client=async_client
)



In [2]:
# Goal is to be able to analyze songs, and analyze images, then render a 10 seconds video of flower sorted by color with the proper background song.

from openagv import AssetBin, SKLoopExecutor, UserInstruction, OTIOTimeline
from openagv.modules.vision import ORVisionAnalyzer

# Define the instruction
instruct = UserInstruction("Make a video of the flowers, each flower 1 second each, but sorted by flower color or maybe like the longer the video gets the wilder the flower is.")

# Setup AssetBin and add assets
ab = SqliteAssetBin("db1.db")
ab.add_wildcard("../assets/*.png")
ab.add_wildcard("../assets/*.jpg")

# Initialize modules
vision_analyzer = ORVisionAnalyzer(client=openai_client, model='mistralai/ministral-8b-2512')
transcriber = DeepgramAnalyzer(api_key=os.getenv("DEEPGRAM_API_KEY"))

timeline = OTIOTimeline()

# Setup Executor
ex = SKLoopExecutor(ab, instruct, chat_completion=chat_completion_service, uses=[vision_analyzer, transcriber, timeline], debug=True)

# Execute
await ex.start()


[INFO] Starting execution...
[DEBUG] Chat History: 2 messages
[INFO] Advancing to step: AssetBin.list_assets
[DEBUG] Invoking AssetBin.list_assets with args: {}
[DEBUG] Result from AssetBin.list_assets: ID: 318ab6f580e22b8733ac4176c3e86111afe162c7d38ba92e5b1320c603e816b7 | File: ../assets\notaflower.png (IMAGE)
ID: bbd4872365cab6c1e5c099270e047a514cc3f15d6902ccdaac88bb01e444b958 | File: ../assets\aarn-giri-3tYZjGSBwbk-unsplash.jpg (IMAGE)
ID: b0ea3882b9aba6470a20dd2f05b77caa18b354226020e9e1ed9d2bb20f4ca806 | File: ../assets\andrew-small-EfhCUc_fjrU-unsplash.jpg (IMAGE)
ID: 6786e06d6f727b5b8cf39ecb41c9f61ca4fe2763ff8ea066d7ce25c3f8845690 | File: ../assets\evie-s-w1JE5duY62M-unsplash.jpg (IMAGE)
ID: b8ef5dd5efbf3c851db53edc34f38eac7f17d3000dfe478ded645d7174498796 | File: ../assets\olia-gozha-9A_peGrSbZc-unsplash.jpg (IMAGE)
ID: 25301e980a702dcb97c2376dae14f9e73b56fee9fb969a464aa4ecf277ce8a31 | File: ../assets\sergey-shmidt-koy6FlCCy5s-unsplash.jpg (IMAGE)
ID: 61415dce13017e2f4dcea0043b1b

In [3]:
ex.asset_bin.json_dump('dump.json')

In [4]:
timeline.to_otio_file('otiodump.otio')

In [5]:
await ex.nudge(UserInstruction("Seems like not all assets are putted into the timeline. Can you ensure everything is on the timeline?"))

[INFO] Nudged with: Seems like not all assets are putted into the timeline. Can you ensure everything is on the timeline?
[DEBUG] Chat History: 19 messages
[INFO] Advancing to step: ChecklistManager.append_new_task
[DEBUG] Invoking ChecklistManager.append_new_task with args: {task_description='Check and ensure all flower assets are added to the timeline.'}
[DEBUG] Result from ChecklistManager.append_new_task: Task added: Check and ensure all flower assets are added to the timeline.
[INFO] Advancing to step: AssetBin.list_assets
[DEBUG] Invoking AssetBin.list_assets with args: {}
[DEBUG] Result from AssetBin.list_assets: ID: 318ab6f580e22b8733ac4176c3e86111afe162c7d38ba92e5b1320c603e816b7 | File: ../assets\notaflower.png (IMAGE)
ID: bbd4872365cab6c1e5c099270e047a514cc3f15d6902ccdaac88bb01e444b958 | File: ../assets\aarn-giri-3tYZjGSBwbk-unsplash.jpg (IMAGE)
ID: b0ea3882b9aba6470a20dd2f05b77caa18b354226020e9e1ed9d2bb20f4ca806 | File: ../assets\andrew-small-EfhCUc_fjrU-unsplash.jpg (IMAGE)

In [6]:
from openagv.renderer import FfmpegOTIORenderer

renderer = FfmpegOTIORenderer()
renderer.set_otio(timeline)
renderer.validate()
renderer.render('test.mp4')

Executing FFmpeg render...
Render complete: test.mp4
